# Preprocessing · PySpark local[6]

Apply a fixed first pass to all 718,330 formal records from Notebook 03. Run in the project kernel.
Use FOT04_PROFILE to select a local Spark profile. The default output is
data/processed/preprocessing/formal_first_pass_v1. Run All verifies and reuses a completed
run with the same inputs and policy. Set FOT04_OUTPUT to a new run directory under
data/processed/preprocessing when comparing a revised policy.

The [historical main-training implementation](../archives/04_preprocessing_main_train_baseline.ipynb)
preserves the earlier scope and source workflow.

This notebook extends the first pass with diagnostics and review-aware reading views.
The formal_first_pass_v1 artifact is reused when present. It records the original fixed
rules and the two structural chunk plans. A later Notebook 05 owns new full-corpus rules
and model-specific exports.


## Start and check local Spark

Set the local profile before starting the kernel. The local6 profile uses six threads.
Choose daily or balanced for a lighter run.

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "configs/preprocessing_notebook.json").is_file()
    ),
    None,
)
if PROJECT_ROOT is None or Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Open the project workspace and select its registered .venv kernel.")

config = json.loads((PROJECT_ROOT / "configs/preprocessing_notebook.json").read_text())
PROFILE = os.environ.get("FOT04_PROFILE", config["default_profile"])
RESOURCE = config["profiles"][PROFILE]
RUNTIME_DIR = PROJECT_ROOT / config["runtime_directory"]
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
STRUCTURE = PROJECT_ROOT / "work_packages/formal_corpus_release_20261117/structure-v2"
OUTPUT_DIR = PROJECT_ROOT / config["output_directory"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = Path(os.environ.get("FOT04_OUTPUT", OUTPUT_DIR / "formal_first_pass_v1")).resolve()
if not OUTPUT.is_relative_to(OUTPUT_DIR) or OUTPUT.parent != OUTPUT_DIR:
    raise ValueError(
        "FOT04_OUTPUT must be a run directory directly inside the preprocessing directory."
    )

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault(
    "PYSPARK_SUBMIT_ARGS", f"--driver-memory {RESOURCE['driver_memory']} pyspark-shell"
)
for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(variable, "1")
print(
    f"Profile {PROFILE}, Python {sys.version.split()[0]}, output {OUTPUT.relative_to(PROJECT_ROOT)}"
)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

if SparkSession.getActiveSession() is not None:
    raise RuntimeError("Start Notebook 04 in a fresh kernel.")

spark = (
    SparkSession.builder.master(RESOURCE["master"])
    .appName("fear-temperature-preprocessing")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.default.parallelism", RESOURCE["threads"])
    .config("spark.sql.shuffle.partitions", RESOURCE["shuffle_partitions"])
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.files.maxPartitionBytes", str(32 * 1024**2))
    .config("spark.local.dir", str(RUNTIME_DIR / "tmp"))
    .config("spark.sql.warehouse.dir", (RUNTIME_DIR / "warehouse").as_uri())
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
if spark.version != config["spark_version"] or spark.sparkContext.master != RESOURCE["master"]:
    raise RuntimeError("Spark configuration differs from the selected notebook profile.")
if (
    spark.range(3).agg(F.sum("id")).first()[0] != 3
    or spark.sparkContext.parallelize([1, 2]).sum() != 3
):
    raise RuntimeError("Spark driver or Python-worker check failed.")
print(f"Spark {spark.version} · {spark.sparkContext.master} · driver and workers verified")

## Findings from Notebooks 01–03 and the first preprocessing pass

[Notebook 01](01_workspace_check.ipynb) verified the local Python kernel, package and 37-row
synthetic fixture. The two pinned model snapshots were unavailable locally.
[Notebook 02](02_synthetic_end_to_end.ipynb) verified lineage, retrieval and aggregation on that
fixture. Its 36 canonical passages retain all 37 lineage rows.

[Notebook 03](03_structure_normalization.ipynb) structured all 718,330 formal records. It
preserved 43 year ranges and found two source-calendar years that differ from their UTC year.
The format rules edited 981 records and flagged 7,163 for review. The near-copy rule surfaced
609 cross-split pairs involving 1,189 records. Blogs averaged 37.2 paragraph spans per record
and forums 3.8, so this pass measures chunking by format and document length.

The first pass applies a small set of auditable formatting and interface rules, protects common
temperature and affect expressions, and compares paragraph-based chunks across the same corpus.
All record and group assignments are inherited. Candidate rules derived from the corpus belong
to the training split. The rules below are fixed, and the resulting counts describe all splits.

The overlap extension from 03 carries both direct evidence and related formal-release
group membership. The diagnostic views below retain those states and show what the existing
first pass changed. Fresh policies for residual whitespace, URL fragments and mixed markup
belong to the next preprocessing stage.


## Rules and expression preservation

Handle CR and CRLF line endings, leading byte-order marks, selected ASCII controls, a short
list of exact semicolon-terminated character references, and unambiguous remaining URL
destinations. Flag paired article footer markers, plain forum quotes, and ambiguous reply
controls for review. Preserve article quotations, forum replies and prose that resembles a
template. Retain unclear URL boundaries and character references in verbatim examples, literal
quotes and tag attributes. Replacement characters, nested references and unusual spellings
remain visible.

The surface audit watches negation, intensifiers, temperature units, selected Australian forms,
emoticons and common emoji sequences. These are string matches. Protected deletions are
blocked, and an edit that reduces a watched count is reverted.
The ledger keeps the exact original slice, replacement and rule. Each edit uses the SHA-bound
Notebook 03 derived text and half-open Unicode code-point offsets. Unedited text is preserved
character for character outside the selected ranges.

The checks are informed by [Unicode Emoji](https://unicode.org/reports/tr51/),
[HTML character reference parsing](https://html.spec.whatwg.org/multipage/parsing.html#named-character-reference-state),
[URI delimiting conventions](https://www.rfc-editor.org/rfc/rfc3986#appendix-C) and
[Hutto and Gilbert on social text cues](https://doi.org/10.1609/icwsm.v8i1.14550).

In [ ]:
import re
from bisect import bisect_left
from hashlib import sha256

# Ordered surface categories for counting. They represent lexical evidence.
CUE_NAMES = ("negation", "intensity", "temperature", "local_usage", "emoticon", "emoji")
CUE_PATTERN = re.compile(
    r"(?P<temperature>(?<!\w)[+−-]?\d+(?:[.,]\d+)?[ \t\u00a0]*(?:"
    r"(?:°|&deg;|&#176;|&#xB0;)[ \t]*[CF]\b|[℃℉]|\b(?:Celsius|Fahrenheit)\b))"
    r"|(?P<negation>\b(?:not|never|no|neither|nor|cannot|[a-z]+n['’]t|"
    r"dont|didnt|doesnt|cant|wont|isnt|wasnt|couldnt|wouldnt|shouldnt)\b)"
    r"|(?P<intensity>\b(?:(?:too|very|so+|really|extremely|bloody)[ \t]+"
    r"(?:hot|cold|warm|freezing|scared|afraid|worried|anxious)|h[o]{3,}t)\b)"
    r"|(?P<local_usage>\b(?:arvo|fair dinkum|fraid|brekkie|mozzie|strewth|"
    r"air[ -]?con|bushfire|realise|colour)\b)"
    r"|(?P<emoticon>(?<!\w)>?[:;=][\-^']?[()DPp/Oo](?!\w))"
    r"|(?P<emoji>[\U0001F1E6-\U0001F1FF]{2}|[0-9#*]\ufe0f?\u20e3|"
    r"[\u2600-\u27bf\U0001F300-\U0001FAFF][\ufe0e\ufe0f]?"
    r"[\U0001F3FB-\U0001F3FF]?(?:\u200d[\u2600-\u27bf\U0001F300-\U0001FAFF]"
    r"[\ufe0e\ufe0f]?[\U0001F3FB-\U0001F3FF]?)*(?:[\U000E0020-\U000E007E]+\U000E007F)?)",
    flags=re.IGNORECASE,
)
CUE_INDEX = {name: index for index, name in enumerate(CUE_NAMES)}

# Exact character references commonly left by HTML-to-text conversion.
ENTITY_VALUES = {
    "amp": "&",
    "nbsp": " ",
    "deg": "°",
    "apos": "'",
    "quot": '"',
    "rsquo": "’",
    "lsquo": "‘",
    "ldquo": "“",
    "rdquo": "”",
}
NUMERIC_VALUES = {ord(value): value for value in ENTITY_VALUES.values()}
NUMERIC_VALUES[160] = " "
ENTITY = re.compile(
    r"&(?P<reference>amp|nbsp|deg|apos|quot|rsquo|lsquo|ldquo|rdquo"
    r"|#[0-9]{1,7}|#[xX][0-9a-fA-F]{1,6});"
)
NESTED_REFERENCE = re.compile(r"(?:[a-zA-Z][a-zA-Z0-9]{1,31}|#[xX]?[a-fA-F0-9]+);")
CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
LINE_END = re.compile(r"\r\n?")
URL = re.compile(
    r"(?<![\w@/.:])(?:https?://|ftp://|www\.)"
    r"(?:[A-Za-z0-9-]+\.)+[A-Za-z][A-Za-z0-9-]*"
    r"(?::[0-9]{1,5})?(?:[/?#][A-Za-z0-9._~:/?@!$&()*+,;=%#\[\]-]*)?",
    re.IGNORECASE,
)
ARTICLE_FORMATS = {"Article", "Blog", "Community publication", "News", "Newsletter"}
ARTICLE_CONTROLS = {
    "advertisement",
    "read more",
    "related articles",
    "related posts",
    "share this article",
    "subscribe now",
}
ARTICLE_ANCHORS = {"share this article", "subscribe now"}
TAIL_LINES = re.compile(r"(?m)^[ \t]*(?P<label>[^\r\n]+?)[ \t]*\r?$")
REPLY_HEAD = re.compile(r"(?m)^.{1,80} said:[ \t]*\r?$")
PLAIN_QUOTE = re.compile(r"(?m)^[ \t]{0,3}>[ \t]+\S")
FORUM_CONTROL = re.compile(r"(?mi)^[ \t]*Click to (?:expand|collapse)(?:\.{3}|…)?[ \t]*\r?$")
PROTECTED_MARKUP = re.compile(r"(?is)\x60|\[(?:code|pre)\b|<(?:code|pre)\b")
TRAILING_EMOTICON = re.compile(r"[:;=]-?[()DPp/Oo](?=[.)\],!?;:]*$)")
URL_REFERENCE = re.compile(r"&(?:[A-Za-z][A-Za-z0-9]*|#[xX]?[A-Fa-f0-9]+);")
HTML_TAG = re.compile(r"""</?[A-Za-z][A-Za-z0-9:_-]*(?=[\s/>])(?:[^<>"']|"[^"]*"|'[^']*')*>""")
QUOTES = {'"': '"', "'": "'", "“": "”", "‘": "’"}
PARAGRAPH_BREAK = re.compile(r"(?:\n[ \t]*){2,}")


def text_hash(text):
    return sha256(text.encode("utf-8")).hexdigest()


def entity_value(text, match):
    raw = match["reference"]
    if raw.startswith(("#x", "#X")):
        value = NUMERIC_VALUES.get(int(raw[2:], 16))
    elif raw.startswith("#"):
        value = NUMERIC_VALUES.get(int(raw[1:]))
    else:
        value = ENTITY_VALUES[raw]
    if value == "&" and NESTED_REFERENCE.match(text, match.end()):
        return None
    return value


def url_end(text, match):
    raw = match.group()
    if TRAILING_EMOTICON.search(raw) or URL_REFERENCE.search(raw):
        return None
    url = raw.rstrip(".,;:!?")
    for opening, closing in (("(", ")"), ("[", "]")):
        while url.endswith(closing) and url.count(closing) > url.count(opening):
            url = url[:-1]
        if url.count(opening) != url.count(closing):
            return None
    end = match.start() + len(url)
    if end == match.end() and end < len(text):
        next_character = text[end]
        if (next_character.isalnum() and not next_character.isascii()) or next_character in "°℃℉":
            return None
    return end


def has_paired_article_footer(text):
    cut = len(text.rstrip())
    # A short final block preceded by a blank line bounds the review candidate.
    where = max(text.rfind("\n\n", 0, cut), text.rfind("\r\n\r\n", 0, cut))
    if where < 0 or cut - where > 240:
        return False
    lines = list(TAIL_LINES.finditer(text, where, cut))
    labels = {line["label"].strip().casefold() for line in lines}
    return ARTICLE_ANCHORS <= labels <= ARTICLE_CONTROLS


def clean_record(text, format_name):
    candidates = [(m.start(), m.end(), "line_ending", "\n") for m in LINE_END.finditer(text)]
    candidates.extend(
        (m.start(), m.end(), "control_character", " ") for m in CONTROL.finditer(text)
    )
    if text.startswith("\ufeff"):
        candidates.append((0, 1, "leading_bom", ""))
    review = set()
    if "\ufffd" in text:
        review.add("replacement_character_review")
    urls = list(URL.finditer(text)) if "://" in text or "www." in text.lower() else []
    if "&" in text:
        tags = [match.span() for match in HTML_TAG.finditer(text)] if "<" in text else []
        for match in ENTITY.finditer(text):
            value = entity_value(text, match)
            if value is None:
                continue
            start, end = match.span()
            if any(tag_start <= start < tag_end for tag_start, tag_end in tags):
                review.add("html_attribute_reference_review")
            elif start > 0 and QUOTES.get(text[start - 1]) == text[end : end + 1]:
                review.add("literal_character_reference_review")
            elif not any(url.start() <= start < url.end() for url in urls):
                candidates.append((start, end, "html_character_reference", value))
    for match in urls:
        end = url_end(text, match)
        if end is None:
            review.add("ambiguous_url_boundary_review")
        else:
            candidates.append((match.start(), end, "url_destination", " "))
    if format_name in ARTICLE_FORMATS and has_paired_article_footer(text):
        review.add("ambiguous_paired_article_footer_review")
    if format_name == "Forum" and ">" in text and PLAIN_QUOTE.search(text):
        review.add("unverified_plain_quote_review")
    if format_name == "Forum" and "click to " in text.lower():
        controls = list(FORUM_CONTROL.finditer(text))
        if controls and REPLY_HEAD.search(text):
            review.add("forum_reply_controls_review")

    cues = [0] * len(CUE_NAMES)
    cue_intervals = []
    for match in CUE_PATTERN.finditer(text):
        category = CUE_INDEX[match.lastgroup]
        cues[category] += 1
        cue_intervals.append(match.span())
    cue_starts = [start for start, _ in cue_intervals]
    has_code = bool(candidates) and PROTECTED_MARKUP.search(text)
    edits = []
    pieces = []
    cursor = 0
    blocked = 0
    for start, end, rule, replacement in sorted(candidates, key=lambda item: (item[0], -item[1])):
        # Retain nested spans in the record that owns their enclosing edit.
        if start < cursor:
            continue
        if rule in {"url_destination", "html_character_reference"} and has_code:
            review.add("structural_rule_in_verbatim_record_review")
            continue
        if rule == "url_destination":
            index = max(0, bisect_left(cue_starts, start) - 1)
            if any(high > start for _, high in cue_intervals[index : bisect_left(cue_starts, end)]):
                blocked += 1
                review.add("deletion_intersects_surface_cue_review")
                continue
        original = text[start:end]
        pieces.extend((text[cursor:start], replacement))
        edits.append((start, end, rule, original, replacement))
        cursor = end
    if edits:
        pieces.append(text[cursor:])
        clean_text = "".join(pieces)
        after_cues = [0] * len(CUE_NAMES)
        for match in CUE_PATTERN.finditer(clean_text):
            after_cues[CUE_INDEX[match.lastgroup]] += 1
        if any(found < expected for found, expected in zip(after_cues, cues, strict=True)):
            review.add("surface_cue_count_loss_review")
            return text, [], cues, cues, sorted(review), blocked + 1
        return clean_text, edits, cues, after_cues, sorted(review), blocked
    return text, [], cues, cues, sorted(review), blocked

## Compare paragraph-based chunking

Use two provisional budgets of 1,200 and 2,400 Unicode code points per chunk. Relative to the
final paragraph offset in Notebook 03, the median was approximately 327 for forums, 2,970 for
blogs and 4,272 for articles. This motivates comparing two budgets and reporting each format.

Greedily combine adjacent whole paragraphs. End offsets tile the cleaned text in order and
include intervening whitespace. Keep an oversized single paragraph in one marked chunk.
A nonblank short entry can form one chunk. Blank entries retain their parent and have zero
chunks. Changed text is segmented again so the two policies address the final text, including
combined emoji sequences within paragraphs. Measure document-length bands, one-chunk rates,
oversized records and chunks, and approximate chunk-length P50/P95. These character budgets
are experimental. See [Unicode text segmentation](https://unicode.org/reports/tr29/).

In [ ]:
# Code-point budgets for comparing a tighter and a wider paragraph plan.
CHUNK_TARGETS = (1200, 2400)


def paragraph_starts(text):
    starts = []
    start = 0
    for boundary in PARAGRAPH_BREAK.finditer(text):
        if text[start : boundary.start()].strip():
            starts.append(start)
        start = boundary.end()
    if text[start:].strip():
        starts.append(start)
    return starts


def chunk_ends(length, starts, target):
    if not starts:
        return []
    ends = []
    chunk_start = 0
    previous_end = 0
    for unit_end in (*starts[1:], length):
        if previous_end > chunk_start and unit_end - chunk_start > target:
            ends.append(previous_end)
            chunk_start = previous_end
        previous_end = unit_end
    ends.append(length)
    return ends


def prepare_record(text, input_sha, format_name, spans):
    clean, edits, before_cues, after_cues, review, blocked = clean_record(text, format_name)
    if edits:
        starts = paragraph_starts(clean)
        output_sha = text_hash(clean)
    else:
        starts = [span[0] for span in spans]
        output_sha = input_sha
    plans = [(target, chunk_ends(len(clean), starts, target)) for target in CHUNK_TARGETS]
    return (
        clean if edits else None,
        output_sha,
        len(text),
        len(clean),
        len(spans),
        len(starts),
        edits,
        before_cues,
        after_cues,
        review,
        blocked,
        plans,
    )

In [ ]:
# Small deterministic checks exercise both edits and common lookalikes.
fixtures = [
    ("Forum", "not only afraid but angry", "not only afraid but angry"),
    ("Forum", "never been so afraid >:( 👩🏽‍🚒 ☀️ 🇦🇺", "never been so afraid >:( 👩🏽‍🚒 ☀️ 🇦🇺"),
    (
        "Forum",
        "I don’t think we are safe. dont 'fraid this arvo.",
        "I don’t think we are safe. dont 'fraid this arvo.",
    ),
    ("Forum", "SOOO hot!!! -3.5 °C and 40&deg;C 🥵", "SOOO hot!!! -3.5 °C and 40°C 🥵"),
    ("Forum", "\ufeffToo hot!\r\n\r\nStill here\x01now", "Too hot!\n\nStill here now"),
    ("Article", "Not afraid &amp; still hot.", "Not afraid & still hot."),
    (
        "Article",
        "40&nbsp;°C &#xB0; &notit; &amp;lt; &lt;tag&gt;",
        "40 °C ° &notit; &amp;lt; &lt;tag&gt;",
    ),
    ("Article", "She wrote: \x60&amp;\x60.", "She wrote: \x60&amp;\x60."),
    ("Article", "I would subscribe now if I could.", "I would subscribe now if I could."),
    ("Article", "Read the interview.\n\nSubscribe now", "Read the interview.\n\nSubscribe now"),
    (
        "Article",
        "Too hot today.\n\nSubscribe now\nShare this article",
        "Too hot today.\n\nSubscribe now\nShare this article",
    ),
    (
        "Article",
        "She wrote “Subscribe now” and “Share this article”.",
        "She wrote “Subscribe now” and “Share this article”.",
    ),
    (
        "Forum",
        "Sam said:\n> 40°C and not afraid\nClick to expand...",
        "Sam said:\n> 40°C and not afraid\nClick to expand...",
    ),
    ("Forum", "[It was too hot](https://example.com/advice).", "[It was too hot]( )."),
    ("Article", "See (https://example.com/path(x)). Too hot.", "See ( ). Too hot."),
    ("Article", "🧊 https://example.com/a/40°C", "🧊 https://example.com/a/40°C"),
    ("Article", "Email a@www.example.com about temp.", "Email a@www.example.com about temp."),
    ("Article", "Literal \ufffd remains visible.", "Literal \ufffd remains visible."),
    ("Article", "Too hot. https://example.com/path:-(", "Too hot. https://example.com/path:-("),
    ("Article", "Too hot. https://example.com/path:D", "Too hot. https://example.com/path:D"),
    (
        "Forum",
        "See https://example.com/thing(not cold).",
        "See https://example.com/thing(not cold).",
    ),
    (
        "Forum",
        "See https://example.com/thing[arvo] next.",
        "See https://example.com/thing[arvo] next.",
    ),
    (
        "Article",
        "See https://example.com/a&emsp;and then go.",
        "See https://example.com/a&emsp;and then go.",
    ),
    (
        "Article",
        "The quoted footer said:\n\nSubscribe now\nShare this article",
        "The quoted footer said:\n\nSubscribe now\nShare this article",
    ),
    (
        "Forum",
        "He said:\r\nClick to expand...\r\nnot afraid",
        "He said:\nClick to expand...\nnot afraid",
    ),
    (
        "Blog",
        'The title is <a title="He said &quot;too hot&quot;">read</a>.',
        'The title is <a title="He said &quot;too hot&quot;">read</a>.',
    ),
    ("Blog", "It literally says '&nbsp;'.", "It literally says '&nbsp;'."),
    ("Forum", "See https://example.com/a:-/", "See https://example.com/a:-/"),
    ("Forum", "See https://example.com/a:-O", "See https://example.com/a:-O"),
]


def replay_edits(source, edits):
    pieces = []
    cursor = 0
    for start, end, _, before, replacement in edits:
        if not cursor <= start < end <= len(source) or source[start:end] != before:
            raise ValueError("Edit evidence does not point to the source text.")
        pieces.extend((source[cursor:start], replacement))
        cursor = end
    pieces.append(source[cursor:])
    return "".join(pieces)


for family, source, expected in fixtures:
    text, edits, retained, observed, _, _ = clean_record(source, family)
    assert text == expected, (source, text, expected)
    assert replay_edits(source, edits) == expected
    assert all(left <= right for left, right in zip(retained, observed, strict=True))
assert "unverified_plain_quote_review" in clean_record(fixtures[12][1], "Forum")[4]
assert "forum_reply_controls_review" in clean_record(fixtures[12][1], "Forum")[4]
assert "ambiguous_url_boundary_review" in clean_record(fixtures[15][1], "Article")[4]
assert "replacement_character_review" in clean_record(fixtures[17][1], "Article")[4]
assert "ambiguous_paired_article_footer_review" in clean_record(fixtures[10][1], "Article")[4]
assert "deletion_intersects_surface_cue_review" in clean_record(fixtures[21][1], "Forum")[4]
assert "forum_reply_controls_review" in clean_record(fixtures[24][1], "Forum")[4]
assert set(
    match.lastgroup for match in CUE_PATTERN.finditer("not afraid, so hot, 40°C, arvo, >:(, 👩🏽‍🚒")
) == set(CUE_NAMES)

# A chunk can be over target if its only paragraph is over target.
chunk_cases = [
    ("", 10, []),
    (" \n\t", 10, []),
    ("👩🏽‍🚒", 2, [4]),
    ("Too hot!", 12, [8]),
    ("aaaa\n\nbbbb", 10, [10]),
    ("aaaa\n\nbbbb", 9, [6, 10]),
    ("xxxxxxxxxxx\n\ny\n\nzz", 7, [13, 18]),
    ("same\n\nsame", 8, [6, 10]),
    ("\n\nshort\n \t\nfine\n", 10, [11, 16]),
]
for text, target, expected in chunk_cases:
    ends = chunk_ends(len(text), paragraph_starts(text), target)
    assert ends == expected, (text, ends, expected)
    assert "".join(text[start:end] for start, end in zip([0, *ends[:-1]], ends, strict=False)) == (
        text if text.strip() else ""
    )
edit_source = "aa\r\n\r\nbb"
result = prepare_record(edit_source, text_hash(edit_source), "Forum", [(0, 2), (6, 8)])
assert result[0] == "aa\n\nbb" and result[-1] == [(1200, [6]), (2400, [6])]
print(f"{len(fixtures)} text cases and {len(chunk_cases) + 1} chunk cases passed.")

# Additional checks for the existing pass's scope and protected text.
scope_examples = [
    ("See 40&nbsp;°C &amp; not afraid.", "See 40 °C & not afraid."),
    ("At &amp;nbsp; I said ‘&nbsp;’.", "At &amp;nbsp; I said ‘&nbsp;’."),
    ("I’m not afraid at 40°C. 👩🏽‍🚒\r\nStill hot.", "I’m not afraid at 40°C. 👩🏽‍🚒\nStill hot."),
    ("  See https:// at the start.\n\n\n", "  See https:// at the start.\n\n\n"),
    (
        "Look at &lt;p&gt; or [quote]Too hot![/quote]",
        "Look at &lt;p&gt; or [quote]Too hot![/quote]",
    ),
]
for original, expected in scope_examples:
    actual, edits, before, after, _, _ = clean_record(original, "Forum")
    assert actual == expected and replay_edits(original, edits) == expected
    assert all(left <= right for left, right in zip(before, after, strict=True))
print(f"{len(scope_examples)} existing-rule scope checks passed.")

## Apply the fixed rules and validate the outputs

Verify the full published metadata and paragraph parts before reading them. Store a compact
record table containing SHA-bound edits, protected-cue counts, review flags and both offset
plans. Unchanged text remains in the verified Notebook 03 input. Use preprocessed_records for
complete cleaned text with publication metadata, chunk_offsets for the relational offset view,
and preprocessed_chunks to request the corresponding substrings. The record evidence and JSON
reports are published together after validation.

In [ ]:
import hashlib
import linecache
import tempfile

import pyarrow.parquet as pq

EXPECTED_FORMAL = 718_330
SPLITS = {"train": 646_414, "validation": 36_110, "test": 35_806}
STRUCTURE_MANIFEST = STRUCTURE / "MANIFEST.json"
structure_manifest = json.loads(STRUCTURE_MANIFEST.read_text())
structure_audit = json.loads((STRUCTURE / "STRUCTURE_AUDIT.json").read_text())


def file_sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


input_manifest_sha = file_sha(STRUCTURE_MANIFEST)
if (
    structure_audit["manifest_sha256"] != input_manifest_sha
    or structure_manifest["run_mode"] != "full"
    or structure_audit["selected_records"] != EXPECTED_FORMAL
    or structure_audit["selected_split_counts"] != SPLITS
):
    raise ValueError("Notebook 04 requires the verified formal structure-v2 snapshot.")


def verify_parquet_inventory(directory, tables):
    checked = 0
    for name, parts in tables.items():
        if not parts or not (directory / name / "_SUCCESS").is_file():
            raise ValueError(f"Missing completed Parquet table {name}.")
        expected = {entry["path"]: entry for entry in parts}
        available = {
            file.relative_to(directory).as_posix(): file
            for file in (directory / name).glob("*.parquet")
        }
        if set(expected) != set(available):
            raise ValueError(f"Unexpected Parquet roster in {name}.")
        for path, file in available.items():
            if file_sha(file) != expected[path]["sha256"]:
                raise ValueError(f"File hash mismatch in {path}.")
            if pq.ParquetFile(file).metadata.num_rows != expected[path]["rows"]:
                raise ValueError(f"File row count mismatch in {path}.")
        checked += len(parts)
    return checked


source_parts = {
    name: structure_manifest["tables"][name]["files"] for name in ("metadata", "paragraph_records")
}
verified_inputs = verify_parquet_inventory(STRUCTURE, source_parts)
print(f"{verified_inputs} Notebook 03 Parquet files verified.")

# The executed notebook cells bind the deterministic text and chunk policy.
policy_sources = [
    "".join(linecache.getlines(function.__code__.co_filename))
    for function in (clean_record, prepare_record)
]
if not all(policy_sources):
    raise RuntimeError("Execute the policy cells in the registered Jupyter kernel.")
policy_sha = text_hash("\n\n".join(policy_sources))
run_identity = {
    "schema_version": 1,
    "input_structure_manifest_sha256": input_manifest_sha,
    "policy_sha256": policy_sha,
    "chunk_targets": list(CHUNK_TARGETS),
}
payload_type = T.StructType.fromDDL("""
    clean_text_override STRING, clean_sha256 STRING,
    input_chars INT, clean_chars INT, input_paragraphs INT, clean_paragraphs INT,
    edits ARRAY<STRUCT<start: INT, end: INT, rule: STRING, before: STRING, replacement: STRING>>,
    protected_input ARRAY<INT>, protected_output ARRAY<INT>, review_flags ARRAY<STRING>,
    blocked_edits INT, plans ARRAY<STRUCT<target: INT, ends: ARRAY<INT>>>
""")
parent_fields = (
    "release_row",
    "native_identity",
    "release_group_id",
    "ml_split",
    "format",
    "source_host",
)
source_rows = spark.read.parquet(str(STRUCTURE / "paragraph_records"))
if OUTPUT.exists():
    prior_manifest = json.loads((OUTPUT / "MANIFEST.json").read_text())
    if any(prior_manifest.get(key) != value for key, value in run_identity.items()):
        raise ValueError("Choose a new FOT04_OUTPUT directory for changed inputs or policy.")
    if file_sha(OUTPUT / "AUDIT.json") != prior_manifest["audit_sha256"]:
        raise ValueError("The existing preprocessing audit failed its hash check.")
    verify_parquet_inventory(OUTPUT, {"records": prior_manifest["records_files"]})
    stage_workspace = None
    run_dir = OUTPUT
    print(f"Reusing verified run {OUTPUT.relative_to(PROJECT_ROOT)}.")
else:
    stage_workspace = tempfile.TemporaryDirectory(prefix=".formal-first-pass-", dir=OUTPUT_DIR)
    run_dir = Path(stage_workspace.name) / "snapshot"
    driver_prepare = F.udf(prepare_record, payload_type, useArrow=False)
    # Carry the compact evidence forward without shuffling bodies or duplicating unchanged text.
    (
        source_rows.select(
            *parent_fields,
            F.col("derived_sha256").alias("input_sha256"),
            F.size("review_flags").alias("upstream_review_flags"),
            driver_prepare(
                "derived_text", "derived_sha256", "format", "paragraphs_in_derived_text"
            ).alias("prepared"),
        )
        .select("*", "prepared.*")
        .drop("prepared")
        .write.mode("errorifexists")
        .option("compression", "zstd")
        .parquet(str(run_dir / "records"))
    )
    print("The full corpus pass has written its staged record evidence.")

records = spark.read.parquet(str(run_dir / "records"))

OVERLAP = STRUCTURE.with_name(f"{STRUCTURE.name}-overlap-v1")
overlap_manifest_path = OVERLAP / "MANIFEST.json"
if not overlap_manifest_path.is_file():
    raise FileNotFoundError(
        "Run the overlap extension in Notebook 03 before opening these quality views."
    )
overlap_manifest = json.loads(overlap_manifest_path.read_text())
overlap_audit_path = OVERLAP / "AUDIT.json"
if overlap_manifest["structure_manifest_sha256"] != input_manifest_sha or overlap_manifest[
    "audit_sha256"
] != file_sha(overlap_audit_path):
    raise ValueError("The overlap evidence is not bound to the verified Notebook 03 snapshot.")
overlap_file_count = verify_parquet_inventory(OVERLAP, overlap_manifest["tables"])
overlap_audit = json.loads(overlap_audit_path.read_text())
overlap_keys = ("release_row", "native_identity", "input_sha256", "release_group_id", "ml_split")
overlap_records = spark.read.parquet(str(OVERLAP / "records")).withColumnRenamed(
    "derived_sha256", "input_sha256"
)
if (
    overlap_records.select("release_row", "native_identity").distinct().count()
    != overlap_audit["direct_or_related_group_records"]
):
    raise ValueError("Overlap evidence is not unique at parent level.")
known = (
    records.select(*overlap_keys)
    .join(F.broadcast(overlap_records.select(*overlap_keys)), list(overlap_keys), "inner")
    .count()
)
if known != overlap_audit["direct_or_related_group_records"] or overlap_records.count() != known:
    raise ValueError("Overlap states do not match the first-pass record identities and hashes.")
print(f"Verified {overlap_file_count} overlap files and {known:,} matching parent keys.")

In [ ]:
# Unpack offset arrays as needed. The stored arrays specify half-open ranges and inherit the parent.
parents_with_plans = (
    records.select(
        *parent_fields,
        "clean_chars",
        "clean_paragraphs",
        F.explode("plans").alias("plan"),
    )
    .select("*", F.col("plan.target").alias("target"), F.col("plan.ends").alias("ends"))
    .drop("plan")
    .withColumn(
        "length_band",
        F.when(F.col("clean_paragraphs") == 0, "blank")
        .when(F.col("clean_chars") <= CHUNK_TARGETS[0], f"up to {CHUNK_TARGETS[0]:,}")
        .when(
            F.col("clean_chars") <= CHUNK_TARGETS[1],
            f"{CHUNK_TARGETS[0] + 1:,} to {CHUNK_TARGETS[1]:,}",
        )
        .otherwise(f"over {CHUNK_TARGETS[1]:,}"),
    )
    .withColumn(
        "sizes",
        F.transform(
            "ends", lambda end, index: end - F.coalesce(F.get("ends", index - 1), F.lit(0))
        ),
    )
    .withColumn("chunks", F.size("sizes"))
    .withColumn("oversized_chunks", F.size(F.filter("sizes", lambda size: size > F.col("target"))))
)


def metric_sections(frame):
    return (
        frame.select(
            "*",
            F.explode(
                F.array(
                    F.struct(
                        F.lit("overall").alias("section"), F.lit("All formats").alias("label")
                    ),
                    F.struct(F.lit("format").alias("section"), F.col("format").alias("label")),
                    F.struct(
                        F.lit("length_band").alias("section"), F.col("length_band").alias("label")
                    ),
                )
            ).alias("section_key"),
        )
        .select("*", "section_key.*")
        .drop("section_key")
    )


def row_dicts(frame, *order):
    return [row.asDict(recursive=True) for row in frame.orderBy(*order).collect()]


def measure_chunk_plans():
    plan_checks = (
        parents_with_plans.agg(
            F.count("*").alias("record_plans"),
            F.count_if(
                ((F.col("clean_paragraphs") == 0) != (F.col("chunks") == 0))
                | (
                    (F.col("clean_paragraphs") > 0)
                    & (
                        (
                            F.aggregate("sizes", F.lit(0), lambda total, size: total + size)
                            != F.col("clean_chars")
                        )
                        | F.exists("sizes", lambda size: size <= 0)
                    )
                )
            ).alias("invalid_coverages"),
        )
        .first()
        .asDict()
    )
    if plan_checks != {
        "record_plans": EXPECTED_FORMAL * len(CHUNK_TARGETS),
        "invalid_coverages": 0,
    }:
        raise ValueError(f"Chunk coverage validation failed: {plan_checks}")
    parent_summary = (
        metric_sections(parents_with_plans)
        .groupBy("section", "label", "target")
        .agg(
            F.count("*").alias("records"),
            F.count_if(F.col("chunks") == 0).alias("blank_records"),
            F.count_if(F.col("chunks") == 1).alias("one_chunk_records"),
            F.count_if(F.col("oversized_chunks") > 0).alias("records_with_oversize"),
            F.sum("chunks").alias("chunks"),
            F.sum("oversized_chunks").alias("oversize_chunks"),
        )
    )
    size_summary = (
        metric_sections(
            parents_with_plans.select(
                "format", "length_band", "target", F.explode("sizes").alias("size")
            )
        )
        .groupBy("section", "label", "target")
        .agg(
            F.percentile_approx("size", [0.5, 0.95], 10_000).alias("chunk_size_p50_p95"),
            F.max("size").alias("max_chunk_chars"),
        )
    )
    report = parent_summary.join(size_summary, ["section", "label", "target"], "left")
    return plan_checks, row_dicts(report, "section", "label", "target")


def measure_cleaning():
    retained = [
        F.sum(F.get("protected_input", i)).alias(f"input_{cue}") for i, cue in enumerate(CUE_NAMES)
    ]
    observed = [
        F.sum(F.get("protected_output", i)).alias(f"output_{cue}")
        for i, cue in enumerate(CUE_NAMES)
    ]
    stats = (
        records.agg(
            F.count("*").alias("records"),
            F.count_if(F.size("edits") > 0).alias("edited_records"),
            F.count_if(F.col("clean_paragraphs") == 0).alias("blank_records"),
            F.count_if(F.col("upstream_review_flags") > 0).alias("upstream_flagged_records"),
            F.count_if(F.size("review_flags") > 0).alias("new_review_records"),
            F.sum("blocked_edits").alias("blocked_edits"),
            F.sum("input_chars").alias("input_codepoints"),
            F.sum("clean_chars").alias("clean_codepoints"),
            F.count_if(
                F.exists(
                    F.arrays_zip("protected_input", "protected_output"),
                    lambda cue: cue.protected_output < cue.protected_input,
                )
            ).alias("records_with_lost_protected_surface_counts"),
            *retained,
            *observed,
        )
        .first()
        .asDict()
    )
    for field, name in (("release_row", "release_rows"), ("native_identity", "native_identities")):
        stats[name] = records.select(field).distinct().count()
    expected_targets = F.array(*[F.lit(target) for target in CHUNK_TARGETS])
    stats["records_with_unexpected_targets"] = records.where(
        F.transform("plans", lambda plan: plan.target) != expected_targets
    ).count()
    if (
        any(
            stats[key] != EXPECTED_FORMAL
            for key in ("records", "release_rows", "native_identities")
        )
        or stats["records_with_unexpected_targets"]
        or stats["records_with_lost_protected_surface_counts"]
    ):
        raise ValueError(f"Record inventory or protection check failed: {stats}")
    splits = {row.ml_split: row["count"] for row in records.groupBy("ml_split").count().collect()}
    if splits != SPLITS:
        raise ValueError(f"Unexpected inherited splits: {splits}")

    edits = (
        records.select("format", "source_host", "release_row", F.explode("edits").alias("edit"))
        .select("*", "edit.*")
        .drop("edit")
    )
    measures = [
        F.count("*").alias("spans"),
        F.countDistinct("release_row").alias("records"),
        F.sum(F.length("before")).alias("source_codepoints"),
        F.sum(F.length("replacement")).alias("replacement_codepoints"),
    ]
    rules = row_dicts(edits.groupBy("rule").agg(*measures), "rule")
    rules_by_source = row_dicts(
        edits.groupBy("format", "source_host", "rule").agg(*measures),
        "format",
        "source_host",
        "rule",
    )
    review = (
        records.select("format", F.explode("review_flags").alias("reason"))
        .groupBy("format", "reason")
        .count()
    )
    return stats, splits, rules, rules_by_source, row_dicts(review, "format", "reason")

In [ ]:
if stage_workspace:
    counts, splits, rules, rules_by_source, review = measure_cleaning()
    plan_checks, plans_report = measure_chunk_plans()
    summary = {
        "input_records": EXPECTED_FORMAL,
        "split_counts": splits,
        "counts": counts,
        "rules": rules,
        "rules_by_source": rules_by_source,
        "review_flags_by_format": review,
        "chunk_invariants": plan_checks,
        "chunk_statistics": plans_report,
        "offsets": {
            "edits": "Half-open Unicode code-point ranges in Notebook 03 derived_text",
            "chunks": "Half-open Unicode code-point ranges in Notebook 04 clean_text",
        },
    }
    audit_file = run_dir / "AUDIT.json"
    audit_file.write_text(json.dumps(summary, ensure_ascii=True, indent=2) + "\n", encoding="utf-8")
    files = [
        {
            "path": path.relative_to(run_dir).as_posix(),
            "rows": pq.ParquetFile(path).metadata.num_rows,
            "sha256": file_sha(path),
        }
        for path in sorted((run_dir / "records").glob("*.parquet"))
    ]
    if (
        sum(part["rows"] for part in files) != EXPECTED_FORMAL
        or not (run_dir / "records/_SUCCESS").is_file()
    ):
        raise ValueError("The staged record table is incomplete.")
    manifest = {
        **run_identity,
        "source_parts_verified": verified_inputs,
        "records_files": files,
        "records_schema": records.schema.jsonValue(),
        "audit_sha256": file_sha(audit_file),
        "spark_version": spark.version,
        "spark_master": spark.sparkContext.master,
    }
    (run_dir / "MANIFEST.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    verify_parquet_inventory(STRUCTURE, source_parts)
    if file_sha(STRUCTURE_MANIFEST) != input_manifest_sha or OUTPUT.exists():
        raise ValueError("The source manifest or output path changed during the run.")
    run_dir.rename(OUTPUT)
    stage_workspace.cleanup()
else:
    summary = json.loads((OUTPUT / "AUDIT.json").read_text())

records = spark.read.parquet(str(OUTPUT / "records"))
source_metadata = spark.read.parquet(str(STRUCTURE / "metadata")).select(
    "release_row",
    "native_identity",
    "source_id",
    "publication_precision",
    "publication_year",
    "publication_date_start",
    "publication_date_end",
    "publication_timestamp_utc",
)


def record_text_view(evidence, paragraphs, metadata):
    return (
        evidence.join(
            paragraphs.select(
                "release_row",
                "native_identity",
                F.col("derived_sha256").alias("input_sha256"),
                "derived_text",
            ),
            ["release_row", "native_identity", "input_sha256"],
            "inner",
        )
        .join(metadata, ["release_row", "native_identity"], "inner")
        .withColumn("clean_text", F.coalesce("clean_text_override", "derived_text"))
        .drop("clean_text_override", "derived_text")
    )


quality_fields = (
    "upstream_review_reasons",
    "review_flags",
    "overlap_status",
    "overlap_rules",
    "overlap_pair_ids",
)


def record_quality_view(confirmed, structure_rows):
    upstream = structure_rows.select(
        "release_row",
        "native_identity",
        F.col("derived_sha256").alias("input_sha256"),
        F.col("review_flags").alias("upstream_review_reasons"),
    )
    return (
        confirmed.join(upstream, ["release_row", "native_identity", "input_sha256"], "left")
        .join(F.broadcast(overlap_records), list(overlap_keys), "left")
        .withColumn(
            "overlap_status",
            F.when(
                F.col("upstream_review_reasons").isNull(), F.lit("structure_reasons_unavailable")
            ).otherwise(
                F.coalesce("overlap_status", F.lit("no_cross_split_evidence_in_current_audit"))
            ),
        )
        .withColumns(
            {
                key: F.coalesce(F.col(key), F.array().cast("array<string>"))
                for key in ("overlap_pair_ids", "overlap_rules")
            }
        )
    )


def chunk_offset_view(quality_records):
    return (
        quality_records.select(
            *parent_fields, *quality_fields, "clean_sha256", F.explode("plans").alias("plan")
        )
        .select(
            *parent_fields,
            *quality_fields,
            "clean_sha256",
            F.col("plan.target").alias("target"),
            F.col("plan.ends").alias("ends"),
            F.posexplode("plan.ends").alias("ordinal", "end"),
        )
        .withColumn("start", F.coalesce(F.get("ends", F.col("ordinal") - 1), F.lit(0)))
        .withColumn("over_target", F.col("end") - F.col("start") > F.col("target"))
        .drop("ends")
    )


quality_records = record_quality_view(records, source_rows)
preprocessed_records = record_text_view(quality_records, source_rows, source_metadata)
chunk_offsets = chunk_offset_view(quality_records)


def chunk_text_view(offsets, parents):
    return (
        offsets.join(
            parents.select("release_row", "native_identity", "clean_sha256", "clean_text"),
            ["release_row", "native_identity", "clean_sha256"],
            "inner",
        )
        .withColumn(
            "chunk_text",
            F.substring("clean_text", F.col("start") + 1, F.col("end") - F.col("start")),
        )
        .drop("clean_text")
    )


preprocessed_chunks = chunk_text_view(chunk_offsets, preprocessed_records)
print(f"Completed evidence at {OUTPUT.relative_to(PROJECT_ROOT)}.")

In [ ]:
edited = records.where((F.size("edits") > 0) & F.exists("plans", lambda p: F.size(p.ends) > 1))
over_target = F.exists(
    "plans",
    lambda p: F.exists(
        F.transform(
            p.ends, lambda end, index: end - F.coalesce(F.get(p.ends, index - 1), F.lit(0))
        ),
        lambda size: size > p.target,
    ),
)
keys = {
    edited.select("release_row").first().release_row,
    records.where(over_target).select("release_row").first().release_row,
    overlap_records.where(F.col("overlap_status") == "direct_evidence").first().release_row,
}
key_filter = F.col("release_row").isin(*keys)
selected_quality = record_quality_view(
    records.where(key_filter),
    F.broadcast(source_rows.where(key_filter)),
)
selected_parents = record_text_view(
    selected_quality,
    F.broadcast(source_rows.where(key_filter)),
    F.broadcast(source_metadata.where(key_filter)),
)
parents = {
    row.release_row: row
    for row in selected_parents.select(
        "release_row",
        "native_identity",
        "ml_split",
        "clean_text",
        "clean_sha256",
        "publication_precision",
        *quality_fields,
    ).collect()
}
assert set(parents) == keys
views = (
    chunk_text_view(chunk_offset_view(selected_quality), F.broadcast(selected_parents))
    .orderBy("release_row", "target", "ordinal")
    .select(
        "release_row",
        "target",
        "native_identity",
        "ml_split",
        "chunk_text",
        "over_target",
        "start",
        "end",
        *quality_fields,
    )
    .collect()
)
assert any(chunk.over_target for chunk in views)
assert any(row.overlap_pair_ids for row in parents.values())
for key, parent in parents.items():
    assert parent.publication_precision and text_hash(parent.clean_text) == parent.clean_sha256
    for target in CHUNK_TARGETS:
        group = [chunk for chunk in views if chunk.target == target and chunk.release_row == key]
        assert all(
            chunk.native_identity == parent.native_identity
            and chunk.ml_split == parent.ml_split
            and all(chunk[field] == parent[field] for field in quality_fields)
            and chunk.chunk_text == parent.clean_text[chunk.start : chunk.end]
            for chunk in group
        )
        assert "".join(chunk.chunk_text for chunk in group) == parent.clean_text
print("Verified edited, oversized and overlap-flagged read views with inherited evidence.")

## Results of the first pass

1. **Operations.** Count all records examined and the records and spans edited. Review the
   lexical checks and inherited overlap evidence alongside the extent of editing.
2. **Text distribution.** Measure format composition and full-record length before and
   after the first pass. Lengths use Unicode code points, and the percentile estimates
   use the same record population.
3. **Chunk comparison.** Compare the two paragraph plans within the same format and
   cleaned parent length band. Oversized parent rates count a parent once per plan.

The three cells reuse the published first-pass records and audit. They read compact fields
and do not create another text or chunk snapshot.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

counts = summary["counts"]
format_edits = (
    records.groupBy("format")
    .agg(
        F.count("*").alias("records"),
        F.count_if(F.size("edits") > 0).alias("edited"),
        F.sum(F.size("edits")).alias("edit_spans"),
        F.count_if((F.col("upstream_review_flags") > 0) | (F.size("review_flags") > 0)).alias(
            "records_with_review_flags"
        ),
    )
    .orderBy(F.desc("records"), "format")
    .toPandas()
)
rule_totals = pd.DataFrame(summary["rules"]).set_index("rule")
rules = (
    rule_totals.reindex(
        [
            "html_character_reference",
            "line_ending",
            "url_destination",
            "control_character",
            "leading_bom",
        ],
        fill_value=0,
    )
    .rename_axis("Rule")
    .reset_index()
)
examined = int(format_edits["records"].sum())
edited_count = int(format_edits["edited"].sum())
span_count = int(format_edits["edit_spans"].sum())
net_codepoints = counts["input_codepoints"] - counts["clean_codepoints"]
if (examined, edited_count, span_count) != (
    summary["input_records"],
    counts["edited_records"],
    int(rules["spans"].sum()),
) or net_codepoints != int((rules["source_codepoints"] - rules["replacement_codepoints"]).sum()):
    raise ValueError("Record and edit summaries do not reconcile with the published first pass.")

display(
    pd.DataFrame(
        [
            ("Records examined", f"{examined:,}"),
            ("Records with an accepted edit", f"{edited_count:,} ({edited_count / examined:.3%})"),
            ("Accepted edit spans", f"{span_count:,}"),
            (
                "Body code points before / after",
                f"{counts['input_codepoints']:,} / {counts['clean_codepoints']:,}",
            ),
            ("Net decrease in body code points", f"{net_codepoints:,}"),
            (
                "Records with a 03 or 04 review reason",
                f"{format_edits['records_with_review_flags'].sum():,}",
            ),
            (
                "Records with a watched expression count decrease",
                f"{counts['records_with_lost_protected_surface_counts']:,}",
            ),
        ],
        columns=["First-pass measure", "Observed value"],
    )
)
display(
    rules.rename(
        columns={
            "records": "Records edited",
            "spans": "Edit spans",
            "source_codepoints": "Input span code points",
            "replacement_codepoints": "Replacement code points",
        }
    )
)
display(
    format_edits.assign(
        edited_share=(100 * format_edits["edited"] / format_edits["records"]).round(3)
    ).rename(
        columns={
            "format": "Format",
            "records": "Examined",
            "edited": "Edited",
            "edit_spans": "Spans",
            "records_with_review_flags": "Distinct records with review reasons",
            "edited_share": "Edited (%)",
        }
    )
)
display(
    pd.DataFrame(
        {
            "Watched expression": CUE_NAMES,
            "Input matches": [counts[f"input_{cue}"] for cue in CUE_NAMES],
            "Output matches": [counts[f"output_{cue}"] for cue in CUE_NAMES],
        }
    )
)
display(
    overlap_records.groupBy("ml_split", "overlap_status")
    .count()
    .orderBy("ml_split", "overlap_status")
    .toPandas()
)

In [ ]:
from matplotlib.ticker import EngFormatter

# The narrow published columns carry one length for every complete parent record.
length_rows = records.select("format", "input_chars", "clean_chars")
probabilities = [0.10, 0.50, 0.90, 0.95, 0.99]
length_quantiles = (
    length_rows.rollup("format")
    .agg(
        F.grouping("format").alias("is_total"),
        F.count("*").alias("records"),
        F.percentile_approx("input_chars", probabilities, 10_000).alias("input_quantiles"),
        F.percentile_approx("clean_chars", probabilities, 10_000).alias("clean_quantiles"),
        F.max("clean_chars").alias("max_clean_chars"),
    )
    .toPandas()
)
length_quantiles["label"] = (
    length_quantiles["format"]
    .where(length_quantiles["is_total"] == 0, "All formats")
    .fillna("Unrecorded format")
)
format_lengths = (
    length_quantiles[length_quantiles["is_total"] == 0]
    .sort_values(["records", "label"], ascending=[False, True])
    .reset_index(drop=True)
)
pooled_lengths = length_quantiles[length_quantiles["is_total"] == 1]
if len(pooled_lengths) != 1 or int(format_lengths["records"].sum()) != examined:
    raise ValueError("Parent length counts do not reconcile across formats.")
if int(pooled_lengths.iloc[0]["records"]) != examined:
    raise ValueError("Overall parent length counts do not reconcile.")

ordered_lengths = pd.concat([pooled_lengths, format_lengths], ignore_index=True)
length_display = pd.DataFrame(
    {
        "Format": ordered_lengths["label"],
        "Records": ordered_lengths["records"],
        "Share of parents (%)": (100 * ordered_lengths["records"] / examined).round(2),
        "Input P50": [value[1] for value in ordered_lengths["input_quantiles"]],
        "Input P95": [value[3] for value in ordered_lengths["input_quantiles"]],
        **{
            f"Output {label}": [value[i] for value in ordered_lengths["clean_quantiles"]]
            for i, label in enumerate(("P10", "P50", "P90", "P95", "P99"))
        },
        "Output max": ordered_lengths["max_clean_chars"],
    }
)
display(length_display)

# Count every parent once per phase, including blank bodies and the open-ended tail.
length_bands = (
    ("0", 0, 0),
    ("1–250", 1, 250),
    ("251–1,200", 251, 1200),
    ("1,201–2,400", 1201, 2400),
    ("2,401–5,000", 2401, 5000),
    ("5,001–10,000", 5001, 10000),
    ("Over 10,000", 10001, None),
)
band_counts = []
for i, (_, lower, upper) in enumerate(length_bands):
    for phase, field in (("input", "input_chars"), ("output", "clean_chars")):
        value = F.col(field)
        in_band = value >= lower if upper is None else value.between(lower, upper)
        band_counts.append(F.count_if(in_band).alias(f"{phase}_{i}"))
size_counts = length_rows.agg(*band_counts).first().asDict()
size_table = pd.DataFrame(
    [
        {
            "Record length (code points)": label,
            "Input records": size_counts[f"input_{i}"],
            "Output records": size_counts[f"output_{i}"],
            "Output share (%)": round(100 * size_counts[f"output_{i}"] / examined, 2),
        }
        for i, (label, _, _) in enumerate(length_bands)
    ]
)
if size_table[["Input records", "Output records"]].sum().tolist() != [examined, examined]:
    raise ValueError("A record length was omitted from the explicit bands.")
display(size_table)

# Show format frequency and the distribution of lengths within each format.
navy = "#142C3C"
teal = "#007E77"
blue = "#3B617F"
amber = "#AB5A20"
muted = "#516574"
grid = "#DCE5E9"
fig, (mix, spread) = plt.subplots(1, 2, figsize=(13, 7.1), sharey=True)
fig.subplots_adjust(left=0.195, right=0.965, top=0.77, bottom=0.17, wspace=0.30)
fig.text(
    0.055, 0.92, "Record formats and full-text lengths", color=navy, fontsize=21, weight="bold"
)
fig.text(
    0.055,
    0.86,
    f"Notebook 04 first pass · {examined:,} formal records",
    color=muted,
    fontsize=11,
)
positions = list(range(len(format_lengths)))
totals = format_lengths["records"].tolist()
mix.barh(positions, totals, height=0.56, color=blue)
mix.set_yticks(positions, format_lengths["label"].tolist())
mix.invert_yaxis()
mix.set_xlim(0, max(totals) * 1.35)
mix.xaxis.set_major_formatter(EngFormatter(sep=""))
mix.set_title("Records by format", loc="left", color=navy, weight="bold")
for y, total in enumerate(totals):
    mix.annotate(
        f"{total:,}",
        (total, y),
        xytext=(5, 0),
        textcoords="offset points",
        va="center",
        fontsize=9,
        color=muted,
    )
mix.set_xlabel("Native records", color=muted)
spread.set_xscale("log")
spread.xaxis.set_major_formatter(EngFormatter(sep=""))
spread.set_title("Cleaned length within format", loc="left", color=navy, weight="bold")
spread.set_xlabel("Unicode code points per parent (log scale)", color=muted)
for position, values in zip(positions, format_lengths["clean_quantiles"], strict=True):
    p10, median, p90, _, _ = values
    # The table retains exact zero quantiles. Draw those at the left boundary of the log axis.
    spread.hlines(position, max(1, p10), max(1, p90), color=teal, linewidth=2.8, alpha=0.6)
    spread.scatter(max(1, median), position, color=teal, s=28, zorder=3)
for axis in (mix, spread):
    axis.set_axisbelow(True)
    axis.grid(axis="x", color=grid, linewidth=0.8)
    axis.spines[["top", "right", "left", "bottom"]].set_visible(False)
    axis.tick_params(axis="both", length=0, labelsize=9, colors=muted)
fig.text(
    0.055,
    0.05,
    "Lines show P10 to P90 and dots show medians. Percentiles use one body per record.",
    color=muted,
    fontsize=9.5,
)
plt.show()
plt.close(fig)

In [ ]:
chunk_summary = pd.DataFrame(summary["chunk_statistics"])
chunk_summary["chunks_per_record"] = (chunk_summary["chunks"] / chunk_summary["records"]).round(2)
chunk_summary["one_chunk_percent"] = (
    100 * chunk_summary["one_chunk_records"] / chunk_summary["records"]
).round(2)
chunk_summary["oversize_record_percent"] = (
    100 * chunk_summary["records_with_oversize"] / chunk_summary["records"]
).round(2)
overall_plans = chunk_summary[chunk_summary["section"] == "overall"].sort_values("target")
format_plans = chunk_summary[chunk_summary["section"] == "format"]
length_plans = chunk_summary[chunk_summary["section"] == "length_band"]

plan_validation = summary["chunk_invariants"]
if (
    overall_plans["target"].tolist() != list(CHUNK_TARGETS)
    or overall_plans["records"].tolist() != [examined] * len(CHUNK_TARGETS)
    or plan_validation["record_plans"] != examined * len(CHUNK_TARGETS)
    or plan_validation["invalid_coverages"] != 0
):
    raise ValueError("The two plan inventories or their recorded coverage do not reconcile.")
for subgroup in (format_plans, length_plans):
    by_target = subgroup.groupby("target")[["records", "chunks"]].sum().sort_index()
    if by_target.to_numpy().tolist() != overall_plans[["records", "chunks"]].to_numpy().tolist():
        raise ValueError("Chunk diagnostic totals do not reconcile.")

print(f"{plan_validation['record_plans']:,} parent–plan combinations checked for coverage.")
display(
    overall_plans[
        [
            "target",
            "records",
            "chunks",
            "chunks_per_record",
            "one_chunk_records",
            "one_chunk_percent",
            "records_with_oversize",
            "oversize_record_percent",
            "oversize_chunks",
            "chunk_size_p50_p95",
            "max_chunk_chars",
        ]
    ].rename(
        columns={
            "target": "Target (code points)",
            "records": "Parent records",
            "chunks": "Chunks",
            "chunks_per_record": "Mean chunks per parent",
            "one_chunk_records": "One-chunk parents",
            "one_chunk_percent": "One-chunk parents (%)",
            "records_with_oversize": "Oversized parents",
            "oversize_record_percent": "Oversized parents (%)",
            "oversize_chunks": "Oversized chunks",
            "chunk_size_p50_p95": "Chunk P50 / P95",
            "max_chunk_chars": "Longest chunk",
        }
    )
)

format_comparison = pd.DataFrame(
    {"Format": format_lengths["label"], "Records": format_lengths["records"]}
)
for target in CHUNK_TARGETS:
    index = format_plans[format_plans["target"] == target].set_index("label")
    format_comparison[f"{target:,} mean chunks"] = format_comparison["Format"].map(
        index["chunks_per_record"]
    )
    format_comparison[f"{target:,} one chunk (%)"] = format_comparison["Format"].map(
        index["one_chunk_percent"]
    )
    format_comparison[f"{target:,} oversized (%)"] = format_comparison["Format"].map(
        index["oversize_record_percent"]
    )
if format_comparison.isna().any().any():
    raise ValueError("A record format is missing its chunk comparison.")
display(format_comparison)

short, wide = CHUNK_TARGETS
all_bands = ("blank", f"up to {short:,}", f"{short + 1:,} to {wide:,}", f"over {wide:,}")
labels = [band for band in all_bands if band in set(length_plans["label"])]
fig, (single, oversized) = plt.subplots(1, 2, figsize=(12.4, 5.6), sharey=True)
fig.subplots_adjust(left=0.18, right=0.95, top=0.71, bottom=0.20, wspace=0.25)
fig.text(
    0.055, 0.92, "Paragraph chunking across record lengths", fontsize=21, weight="bold", color=navy
)
fig.text(
    0.055,
    0.85,
    "Each rate uses the parent records in that cleaned-length band.",
    fontsize=11,
    color=muted,
)
for axis, metric, title in (
    (single, "one_chunk_percent", "Parents with a single chunk"),
    (oversized, "oversize_record_percent", "Parents with an oversized chunk"),
):
    for target, color, offset in zip(CHUNK_TARGETS, (blue, amber), (-0.18, 0.18), strict=True):
        values = (
            length_plans[length_plans["target"] == target]
            .set_index("label")
            .loc[labels, metric]
            .tolist()
        )
        bars = axis.barh(
            [i + offset for i in range(len(labels))],
            values,
            height=0.31,
            color=color,
            label=f"{target:,} code points",
        )
        axis.bar_label(
            bars, labels=[f"{value:g}%" for value in values], padding=5, color=muted, fontsize=9
        )
    axis.set_yticks(range(len(labels)), [label.capitalize() for label in labels])
    axis.set_xlim(0, 115)
    axis.set_xticks([0, 25, 50, 75, 100])
    axis.set_xlabel("Parent records (%)", color=muted)
    axis.set_title(title, loc="left", color=navy, weight="bold")
    axis.set_axisbelow(True)
    axis.grid(axis="x", color=grid, linewidth=0.8)
    axis.spines[["top", "right", "left", "bottom"]].set_visible(False)
    axis.tick_params(axis="both", length=0, labelsize=9, colors=muted)
single.invert_yaxis()
fig.legend(
    *single.get_legend_handles_labels(),
    loc="upper left",
    bbox_to_anchor=(0.05, 0.82),
    ncol=2,
    frameon=False,
)
fig.text(
    0.055,
    0.06,
    "An oversized single paragraph remains intact. The two plans use character budgets.",
    fontsize=9.5,
    color=muted,
)
plt.show()
plt.close(fig)

## Bind the first pass to the overlap evidence

Publish a separate quality receipt for this first-pass and overlap snapshot pair. It records
the overlap states by split and the frequency of each Notebook 03 review reason. The original
first-pass manifest continues to identify the text and chunk policy.

The complete URL rule applied zero edits on this snapshot. Its scope requires a recognized
host-bearing destination. Notebook 05 tracks scheme-shaped fragments separately.

In [ ]:
overlap_status_counts = [
    row.asDict()
    for row in quality_records.groupBy("ml_split", "overlap_status")
    .count()
    .orderBy("ml_split", "overlap_status")
    .collect()
]
upstream_reason_counts = [
    row.asDict()
    for row in source_rows.select(F.explode("review_flags").alias("reason"))
    .groupBy("reason")
    .count()
    .orderBy("reason")
    .collect()
]
if sum(row["count"] for row in overlap_status_counts) != EXPECTED_FORMAL or any(
    row["overlap_status"] == "structure_reasons_unavailable" for row in overlap_status_counts
):
    raise ValueError("The overlap and review disposition does not cover the formal roster.")
quality_binding = {
    "schema_version": 1,
    "first_pass_manifest_sha256": file_sha(OUTPUT / "MANIFEST.json"),
    "structure_manifest_sha256": input_manifest_sha,
    "overlap_manifest_sha256": file_sha(overlap_manifest_path),
    "overlap_audit_sha256": file_sha(overlap_audit_path),
    "formal_records": EXPECTED_FORMAL,
    "overlap_status_by_split": overlap_status_counts,
    "overlap_audit_scope": overlap_audit["setting"],
    "structure_review_reason_counts": upstream_reason_counts,
    "url_destination_edits": next(
        (row["spans"] for row in summary["rules"] if row["rule"] == "url_destination"), 0
    ),
    "url_rule_scope": "Recognized complete host-bearing destinations",
}
binding = OUTPUT.with_name(f"{OUTPUT.name}_quality_v1.json")
if binding.is_file():
    if quality_binding != json.loads(binding.read_text()):
        raise ValueError(
            "The saved quality binding disagrees with the first-pass or overlap inputs."
        )
else:
    with tempfile.TemporaryDirectory(prefix=".first-pass-quality-", dir=OUTPUT.parent) as stage:
        staged_binding = Path(stage) / binding.name
        staged_binding.write_text(json.dumps(quality_binding, indent=2) + "\n")
        binding.hardlink_to(staged_binding)
print(f"Verified the quality binding at {binding.relative_to(PROJECT_ROOT)}.")

The Spark session stays available for inspecting the two views. Use the kernel's shutdown command after inspection.